## Entendimiento del problema

### By:
Maria Camila Aristizábal Aguirre

### Date:
2026-08-15

### Description:

Primera ejecución del Proyecto 1 - Dataset estático.
Respuesta a las preguntas del negocio y carga de datos


## 📊 Preguntas y respuestas

## 1. ¿Cuál es el objetivo del problema?

Predecir de forma diagnóstica si una paciente tiene diabetes o no, a partir de ocho mediciones clínicas (número de embarazos, glucosa, presión sanguínea, espesor del pliegue cutáneo, insulina, índice de masa corporal, función de pedigrí de diabetes y edad). La población está restringida a mujeres de al menos 21 años. La variable objetivo (Outcome) es binaria: 1 si la paciente tiene diabetes, 0 si no.

## 2. ¿Cómo se usará su solución?

El modelo funcionaría como una herramienta de apoyo al tamizaje (screening) clínico, no como diagnóstico definitivo. Por ejemplo, podría alimentar un sistema que marque pacientes de alto riesgo para que un profesional de salud ordene pruebas confirmatorias (glucosa en ayunas, prueba de tolerancia oral a la glucosa, HbA1c). La salida (probabilidad o etiqueta 0/1) sería un insumo para la decisión médica, no un reemplazo de ella.

## 3. ¿Cuáles son las soluciones actuales (si las hay)?

No hay solución actual documentada dentro del propio dataset, pero en el dominio existen dos tipos de "solucines" a considerar:

Diagnóstico clínico estándar: pruebas de laboratorio (glucosa en ayunas ≥126 mg/dL, prueba de tolerancia oral a la glucosa, HbA1c ≥6.5%) según criterios de la ADA/OMS, más el juicio clínico del médico.

Este mismo dataset como benchmark de ML: el Pima Indians Diabetes Dataset (origen NIDDK) es uno de los conjuntos de datos más utilizados en la literatura de aprendizaje automático para clasificación binaria; existen decenas de modelos publicados (regresión logística, árboles, SVM, redes neuronales) con exactitudes reportadas típicamente entre 70% y 80%. Estos sirven como referencia (baseline) de lo que es razonable esperar.

## 4. ¿Cómo se debe enmarcar este problema?

Supervisado: existe una variable objetivo etiquetada (Outcome).
Clasificación binaria (no regresión): la salida es una categoría, 0 o 1.

Fuera de línea / por lotes (batch): el dataset es una tabla estática de tamaño fijo, sin indicios de flujo continuo de nuevos datos ni necesidad de actualización en tiempo real. Un reentrenamiento periódico (por lotes) es suficiente; no se justifica aprendizaje en línea a menos que el sistema se integre después a un flujo hospitalario en vivo.

Basado en modelo (model-based) más que en instancias: con menos de 1000 registros y 8 variables, tiene sentido ajustar un modelo paramétrico o un árbol/ensamble en lugar de comparar contra instancias almacenadas (k-NN sería viable pero menos habitual como primera aproximación).

En síntesis, el problema queda enmarcado como: aprendizaje supervisado, de clasificación binaria, fuera de línea (por lotes) y basado en modelo.

## 5. ¿Cómo se debe medir el desempeño?

El accuracy simple no es suficiente por dos razones observadas en los datos: hay desbalance de clases moderado y, en un contexto médico, los dos tipos de error no cuestan lo mismo. Una primera intuición razonable:

Recall / sensibilidad de la clase 1 (diabetes) como métrica principal: minimizar falsos negativos (pacientes diabéticas clasificadas como sanas) es más crítico que minimizar falsos positivos, porque un falso negativo retrasa el diagnóstico y tratamiento.

Precision y F1-score como métricas de apoyo, para no sacrificar demasiada precisión al optimizar recall.

AUC-ROC para evaluar la capacidad discriminativa del modelo independientemente del umbral de decisión.

Matriz de confusión completa para inspección cualitativa.

## 6. ¿La medida de desempeño está alineada con el objetivo del problema?

Sí, si se prioriza recall/sensibilidad sobre accuracy: el objetivo real (apoyar el diagnóstico temprano) se traduce directamente en "no dejar pasar casos positivos". Usar solo accuracy estaría mal alineado, porque un modelo que siempre prediga "0" (no diabetes) ya alcanzaría cerca de 63-65% de accuracy en este dataset sin ser clínicamente útil.

## 7. ¿Cuál sería el desempeño mínimo necesario para alcanzar el objetivo?

Como referencia: dado que un clasificador trivial que prediga siempre la clase mayoritaria ya obtiene ~63% de accuracy, cualquier modelo debería superar claramente ese piso. Tomando como referencia la literatura publicada sobre este mismo dataset (accuracy ~75-80%, AUC ~0.80-0.85), un desempeño mínimo razonable para considerar la solución útil sería algo como recall ≥ 0.75-0.80 en la clase diabetes y AUC-ROC ≥ 0.80, aceptando que estas cifras deberían validarse con el equipo clínico, no fijarse unilateralmente desde el análisis de datos.

## 8. ¿Cuáles son los problemas parecidos? ¿Se pueden reutilizar experiencias o herramientas ya creadas?

Sí. Este es un problema clásico de clasificación binaria tabular con datos clínicos, muy similar a: predicción de enfermedad cardiaca, predicción de cáncer (Wisconsin Breast Cancer Dataset), predicción de readmisión hospitalaria, scoring de riesgo crediticio (misma familia de técnicas aunque dominio distinto). 

Se puede reutilizar directamente: pipelines estándar de scikit-learn (imputación, escalado, regresión logística/árboles/gradient boosting), técnicas de manejo de clases desbalanceadas (class_weight, SMOTE), y el hecho de que el propio dataset es uno de los más estudiados en ML educativo, por lo que hay abundante literatura y notebooks de referencia para comparar resultados.

## 9. ¿Hay experiencia del problema disponible?

Sí, en dos niveles: (a) experiencia clínica amplia sobre los factores de riesgo de diabetes tipo 2 (obesidad/BMI alto, edad, antecedentes familiares —reflejado en DiabetesPedigreeFunction—, glucosa elevada, número de embarazos en mujeres), y (b) experiencia en ciencia de datos, ya que el Pima Indians Diabetes Dataset lleva décadas siendo usado como caso de estudio, con numerosos análisis, artículos y tutoriales publicados que documentan qué variables son más predictivas y qué problemas de calidad de datos tiene (ver supuestos abajo).

## 10. (Importante) ¿Cómo se puede resolver el problema manualmente?

Un médico resolvería el problema manualmente aplicando reglas clínicas conocidas en lugar de un modelo estadístico: por ejemplo, glucosa en ayunas ≥126 mg/dL o glucosa a 2 horas en prueba de tolerancia oral ≥200 mg/dL sugiere diabetes; BMI ≥30 (obesidad) y edad avanzada aumentan el riesgo; antecedentes familiares fuertes (reflejados indirectamente en DiabetesPedigreeFunction) también. 

En la práctica, un profesional combinaría estos umbrales con el cuadro clínico completo del paciente y, si hay duda, ordenaría una prueba de laboratorio confirmatoria. Este razonamiento manual es exactamente lo que el modelo intentará aproximar y automatizar a partir de los patrones en los datos.

## 11. Listado de supuestos hasta este momento

El dataset corresponde al conocido Pima Indians Diabetes Dataset (NIDDK), aunque con modificaciones respecto a la versión original (más filas, valores faltantes y errores de formato introducidos), probablemente para fines del ejercicio.

Hay 185 filas duplicadas (aprox. 19% del total) — se asume que deben eliminarse antes de modelar.

Hay valores faltantes explícitos (celdas vacías) en todas las columnas, entre 2 y 19 registros según la columna; en Outcome hay 19 filas sin etiqueta, que probablemente deban descartarse ya que no sirven para entrenamiento supervisado.

Hay ceros que en realidad representan datos faltantes, no valores reales: Insulin tiene 485 ceros (~49%), SkinThickness 298 ceros (~30%), BloodPressure 47, BMI 14, Glucose 8. Fisiológicamente, una presión sanguínea, glucosa o BMI de 0 no es posible en una persona viva, por lo que se asume que estos ceros son "faltantes disfrazados" y deberán tratarse como NaN e imputarse, no como valores válidos.

La columna DiabetesPedigreeFunction tiene un problema de formato: en el dataset original sus valores van de ~0.08 a ~2.42. Aquí, 894 de 989 valores no nulos son mayores a 3 (hasta 2329), lo que sugiere que a la mayoría de los registros se les "perdió" el punto decimal (por ejemplo, 627 en vez de 0.627). Se asume que es un error de formato/importación a corregir dividiendo por 1000 los valores afectados, no una variable con otra escala.

Las clases están moderadamente desbalanceadas: de los registros con etiqueta válida, ~63.8% son Outcome=0 (no diabetes) y ~36.2% son Outcome=1 (diabetes). No es un desbalance extremo, pero sí suficiente para que el accuracy simple sea engañoso.

Se asume que la población de origen (mujeres ≥21 años, herencia Pima) puede no generalizar directamente a otras poblaciones; el modelo resultante aplicaría, en principio, solo a un grupo demográfico similar al de entrenamiento.

Se asume que el uso previsto es de apoyo al diagnóstico (screening), no diagnóstico automático definitivo, dado que no se especifica lo contrario.

## 12. ¿Cuál es la fuente de los datos?

Informacion.txt no lo indica explícitamente, pero por su estructura y las variables incluidas, se asume que es una variante del Pima Indians Diabetes Dataset, originado en el National Institute of Diabetes and Digestive and Kidney Diseases (NIDDK) de EE. UU., ampliamente redistribuido a través de plataformas como Kaggle y el UCI Machine Learning Repository con fines educativos.

## 13. ¿Cómo se actualizan los datos?

Dado que se trata de un archivo .csv estático entregado como parte de un trabajo académico, se asume que no hay actualización: es una foto fija de datos históricos para fines de entrenamiento y evaluación, no un flujo de datos vivo.

## 14. ¿Cada cuánto tiempo se actualizan los datos?

Consistente con el punto anterior, se asume que no aplica (frecuencia de actualización = nunca / no definida) para este trabajo, ya que no existe una fuente en producción ni un pipeline de datos descrito. Si el modelo se llevara a un entorno real, esta pregunta habría que respondérsela al equipo de TI/clínico responsable del sistema de origen de los datos.

## 🔍 Carga y exploración inicial de los datos

Verificación con código de los supuestos descritos arriba.

In [1]:
import pandas as pd

df = pd.read_csv("../../data/01_raw/diabetes.csv")

print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}")
df.head()

Filas: 994, Columnas: 9


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6.0,148.0,72.0,35.0,0.0,33.6,627.0,50.0,1.0
1,1.0,85.0,66.0,29.0,0.0,26.6,351.0,31.0,0.0
2,8.0,183.0,64.0,0.0,0.0,23.3,672.0,32.0,1.0
3,1.0,89.0,66.0,23.0,94.0,28.1,167.0,21.0,0.0
4,0.0,137.0,40.0,35.0,168.0,43.1,2288.0,33.0,1.0


In [2]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 994 entries, 0 to 993
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               986 non-null    float64
 1   Glucose                   990 non-null    float64
 2   BloodPressure             989 non-null    float64
 3   SkinThickness             987 non-null    float64
 4   Insulin                   986 non-null    float64
 5   BMI                       992 non-null    float64
 6   DiabetesPedigreeFunction  989 non-null    float64
 7   Age                       991 non-null    float64
 8   Outcome                   975 non-null    float64
dtypes: float64(9)
memory usage: 70.0 KB


In [3]:
n_duplicados = df.duplicated().sum()
pct_duplicados = n_duplicados / len(df) * 100

print(f"Filas duplicadas: {n_duplicados} ({pct_duplicados:.1f}% del total)")


Filas duplicadas: 185 (18.6% del total)


In [4]:
nulos = df.isna().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)

print("Valores nulos por columna:")
print(nulos)
print(f"\nFilas sin 'Outcome' (no sirven para entrenar): {df['Outcome'].isna().sum()}")


Valores nulos por columna:
Outcome                     19
Pregnancies                  8
Insulin                      8
SkinThickness                7
BloodPressure                5
DiabetesPedigreeFunction     5
Glucose                      4
Age                          3
BMI                          2
dtype: int64

Filas sin 'Outcome' (no sirven para entrenar): 19


In [5]:
columnas_sin_cero_valido = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
]

print("Ceros 'disfrazados' de faltante por columna:")
for col in columnas_sin_cero_valido:
    n_ceros = (df[col] == 0).sum()
    pct = n_ceros / len(df) * 100
    print(f"  {col}: {n_ceros} ({pct:.1f}%)")


Ceros 'disfrazados' de faltante por columna:
  Glucose: 8 (0.8%)
  BloodPressure: 47 (4.7%)
  SkinThickness: 298 (30.0%)
  Insulin: 485 (48.8%)
  BMI: 14 (1.4%)


In [6]:
dpf_no_nulo = df["DiabetesPedigreeFunction"].notna().sum()
dpf_mayor_a_3 = (df["DiabetesPedigreeFunction"] > 3).sum()

print(f"Valores no nulos: {dpf_no_nulo}")
print(f"Valores > 3 (posible punto decimal perdido): {dpf_mayor_a_3}")
df["DiabetesPedigreeFunction"].describe()


Valores no nulos: 989
Valores > 3 (posible punto decimal perdido): 894


count     989.000000
mean      430.780536
std       338.208293
min         0.100000
25%       209.000000
50%       337.000000
75%       591.000000
max      2329.000000
Name: DiabetesPedigreeFunction, dtype: float64

In [7]:
outcome_counts = df["Outcome"].value_counts(dropna=True)
outcome_pct = df["Outcome"].value_counts(normalize=True, dropna=True) * 100

print(outcome_counts)
print()
print(outcome_pct.round(1))


Outcome
0.0    630
1.0    345
Name: count, dtype: int64

Outcome
0.0    64.6
1.0    35.4
Name: proportion, dtype: float64
